In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "/kaggle/input/competitions/shopee-product-matching"

train = pd.read_csv(f"{DATA_PATH}/train.csv")

print(train.shape)
print(train.columns.tolist())

(34250, 5)
['posting_id', 'image', 'image_phash', 'title', 'label_group']


# Splitting product grps (labl_grp) so that final test eval will be easier

In [2]:
from sklearn.model_selection import train_test_split

groups = train["label_group"].unique()

train_groups, temp_groups = train_test_split(
    groups,
    test_size=0.30,
    random_state=42
)

val_groups, test_groups = train_test_split(
    temp_groups,
    test_size=0.50,
    random_state=42
)

train_data = train[train["label_group"].isin(train_groups)].reset_index(drop=True)
val_data = train[train["label_group"].isin(val_groups)].reset_index(drop=True)
test_data = train[train["label_group"].isin(test_groups)].reset_index(drop=True)

print("Train listings:", len(train_data))
print("Validation listings:", len(val_data))
print("Test listings:", len(test_data))

print("\nTrain groups:", train_data["label_group"].nunique())
print("Validation groups:", val_data["label_group"].nunique())
print("Test groups:", test_data["label_group"].nunique())

Train listings: 23629
Validation listings: 5189
Test listings: 5432

Train groups: 7709
Validation groups: 1652
Test groups: 1653


# Create matching pairs

In [3]:
def create_pairs(data, seed=42):
    rng = np.random.default_rng(seed)

    positive_pairs = []
    negative_pairs = []

    # Create positive pairs
    for group, group_data in data.groupby("label_group"):

        if len(group_data) >= 2:
            i, j = rng.choice(len(group_data), size=2, replace=False)

            positive_pairs.append([
                group_data.iloc[i]["title"],
                group_data.iloc[j]["title"],
                1
            ])

    # Create negative pairs
    groups = data["label_group"].unique()

    for _ in range(len(positive_pairs)):

        group1, group2 = rng.choice(groups, size=2, replace=False)

        listing1 = data[data["label_group"] == group1].sample(
            1,
            random_state=int(rng.integers(100000))
        ).iloc[0]

        listing2 = data[data["label_group"] == group2].sample(
            1,
            random_state=int(rng.integers(100000))
        ).iloc[0]

        negative_pairs.append([
            listing1["title"],
            listing2["title"],
            0
        ])

    pairs = pd.DataFrame(
        positive_pairs + negative_pairs,
        columns=["title1", "title2", "label"]
    )

    return pairs.sample(frac=1, random_state=seed).reset_index(drop=True)

# We should get like balanced 0 and 1 classes

In [4]:
train_pairs = create_pairs(train_data, seed=42)
val_pairs = create_pairs(val_data, seed=43)
test_pairs = create_pairs(test_data, seed=44)

print("Train pairs:", len(train_pairs))
print("Validation pairs:", len(val_pairs))
print("Test pairs:", len(test_pairs))

print("\nTrain labels:")
print(train_pairs["label"].value_counts())

print("\nValidation labels:")
print(val_pairs["label"].value_counts())

print("\nTest labels:")
print(test_pairs["label"].value_counts())

Train pairs: 15418
Validation pairs: 3304
Test pairs: 3306

Train labels:
label
0    7709
1    7709
Name: count, dtype: int64

Validation labels:
label
0    1652
1    1652
Name: count, dtype: int64

Test labels:
label
0    1653
1    1653
Name: count, dtype: int64


In [5]:
print("MATCH EXAMPLES")
print(
    train_pairs[train_pairs["label"] == 1]
    .head(5)
    .to_string(index=False)
)

print("\nNON-MATCH EXAMPLES")
print(
    train_pairs[train_pairs["label"] == 0]
    .head(5)
    .to_string(index=False)
)

MATCH EXAMPLES
                                                 title1                                                   title2  label
     (COD) JAKET HOODIE FREE FIRE / SWEATER GAME GARENA          JAKET HOODIE GAME FREE FIRE / JAKET HOODIE GAME      1
                          Buah segar - lemon Impor /pcs                         lemon california 1kg isi 7-9buah      1
Setelan Anak Cowok Merah Octopus Size 1, 2, 3 thn Murah     Setelan anak cowo merah gurita size 1,2,3tahun murah      1
         Jamu montok payudara JK (bisa bayar di tempat)                                     Jamu Montok Payudara      1
         Redmi Note 8 Pro 6/64 & 6/128 Gb Garansi Resmi Redmi Note 8 Pro 6/128 GB GARANSI RESMI XIAOMI INDONESIA      1

NON-MATCH EXAMPLES
                                                                                                                    title1                                                                              title2  label
\xe2\x9c\xa7YUKSHOPING\xe2\x9c\

# Baseline — Word TF-IDF

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Use only training titles to learn the vocabulary
train_titles = pd.concat([
    train_pairs["title1"],
    train_pairs["title2"]
])

vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2
)

vectorizer.fit(train_titles)

print("Number of TF-IDF features:", len(vectorizer.vocabulary_))

Number of TF-IDF features: 68747


# Baseline — TF-IDF similarity

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

# Validation vectors
val_vec1 = vectorizer.transform(val_pairs["title1"])
val_vec2 = vectorizer.transform(val_pairs["title2"])

# Similarity for each pair
val_similarity = cosine_similarity(
    val_vec1, val_vec2
).diagonal()

print(val_similarity[:10])

[0.         1.         0.         0.         0.72056296 0.59095637
 0.4911935  0.78018468 0.62089362 0.48237963]


In [9]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_val = val_pairs["label"].values

thresholds = np.arange(0.10, 0.91, 0.05)
results = []

for threshold in thresholds:
    predictions = (val_similarity >= threshold).astype(int)

    precision = precision_score(y_val, predictions, zero_division=0)
    recall = recall_score(y_val, predictions, zero_division=0)
    f1 = f1_score(y_val, predictions, zero_division=0)

    results.append([threshold, precision, recall, f1])

threshold_results = pd.DataFrame(
    results,
    columns=["threshold", "precision", "recall", "f1"]
)

print(threshold_results)

    threshold  precision    recall        f1
0        0.10   0.995513  0.940073  0.966999
1        0.15   0.998015  0.912833  0.953525
2        0.20   0.999310  0.877119  0.934236
3        0.25   0.999277  0.836562  0.910708
4        0.30   0.999228  0.783293  0.878181
5        0.35   0.999175  0.733051  0.845670
6        0.40   0.999105  0.675545  0.806067
7        0.45   0.999022  0.618644  0.764112
8        0.50   0.998924  0.561743  0.719101
9        0.55   0.998798  0.503027  0.669082
10       0.60   0.998626  0.440073  0.610924
11       0.65   0.998410  0.380145  0.550636
12       0.70   0.998131  0.323245  0.488340
13       0.75   0.997748  0.268160  0.422710
14       0.80   0.997183  0.214286  0.352765
15       0.85   0.996644  0.179782  0.304615
16       0.90   0.995413  0.131356  0.232086


In [10]:
best_threshold = threshold_results.loc[
    threshold_results["f1"].idxmax(),
    "threshold"
]

print("Best threshold:", best_threshold)

Best threshold: 0.1


In [11]:
test_vec1 = vectorizer.transform(test_pairs["title1"])
test_vec2 = vectorizer.transform(test_pairs["title2"])

test_similarity = cosine_similarity(
    test_vec1, test_vec2
).diagonal()

y_test = test_pairs["label"].values

test_predictions = (
    test_similarity >= best_threshold
).astype(int)

print("TF-IDF BASELINE")
print("Threshold:", best_threshold)
print("Precision:", precision_score(y_test, test_predictions, zero_division=0))
print("Recall:", recall_score(y_test, test_predictions, zero_division=0))
print("F1:", f1_score(y_test, test_predictions, zero_division=0))

TF-IDF BASELINE
Threshold: 0.1
Precision: 0.9942711648631445
Recall: 0.9449485783424078
F1: 0.9689826302729528


# Experiment 1 — Character TF-IDF

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2
)

char_train_titles = pd.concat([
    train_pairs["title1"],
    train_pairs["title2"]
])

char_vectorizer.fit(char_train_titles)

print("Number of character features:",
      len(char_vectorizer.vocabulary_))

Number of character features: 213982


In [13]:
# Validation similarity

val_char1 = char_vectorizer.transform(val_pairs["title1"])
val_char2 = char_vectorizer.transform(val_pairs["title2"])

val_char_similarity = cosine_similarity(
    val_char1,
    val_char2
).diagonal()

# Find best threshold

char_results = []

for threshold in np.arange(0.10, 0.91, 0.05):

    predictions = (val_char_similarity >= threshold).astype(int)

    precision = precision_score(
        val_pairs["label"], predictions, zero_division=0
    )
    recall = recall_score(
        val_pairs["label"], predictions, zero_division=0
    )
    f1 = f1_score(
        val_pairs["label"], predictions, zero_division=0
    )

    char_results.append([
        threshold, precision, recall, f1
    ])

char_results = pd.DataFrame(
    char_results,
    columns=["threshold", "precision", "recall", "f1"]
)

best_char_threshold = char_results.loc[
    char_results["f1"].idxmax(), "threshold"
]

print(char_results)
print("\nBest threshold:", best_char_threshold)

    threshold  precision    recall        f1
0        0.10   0.994347  0.958232  0.975956
1        0.15   0.998705  0.933414  0.964956
2        0.20   0.998655  0.898910  0.946161
3        0.25   0.999292  0.854722  0.921370
4        0.30   0.999237  0.792978  0.884239
5        0.35   0.999161  0.720944  0.837553
6        0.40   0.999077  0.654964  0.791225
7        0.45   0.998963  0.582930  0.736239
8        0.50   0.998804  0.505448  0.671222
9        0.55   0.998630  0.441283  0.612091
10       0.60   0.998403  0.378329  0.548727
11       0.65   0.998092  0.316586  0.480699
12       0.70   0.997630  0.254843  0.405979
13       0.75   0.997085  0.207022  0.342857
14       0.80   0.996491  0.171913  0.293237
15       0.85   0.995434  0.131961  0.233030
16       0.90   1.000000  0.095642  0.174586

Best threshold: 0.1


In [14]:
# Final test evaluation

test_char1 = char_vectorizer.transform(test_pairs["title1"])
test_char2 = char_vectorizer.transform(test_pairs["title2"])

test_char_similarity = cosine_similarity(
    test_char1,
    test_char2
).diagonal()

char_predictions = (
    test_char_similarity >= best_char_threshold
).astype(int)

print("CHARACTER TF-IDF")
print("Threshold:", best_char_threshold)
print("Precision:", precision_score(
    test_pairs["label"], char_predictions, zero_division=0
))
print("Recall:", recall_score(
    test_pairs["label"], char_predictions, zero_division=0
))
print("F1:", f1_score(
    test_pairs["label"], char_predictions, zero_division=0
))

CHARACTER TF-IDF
Threshold: 0.1
Precision: 0.9956222639149468
Recall: 0.9630973986690865
F1: 0.9790897908979089


# Experiment 2 — Multilingual Sentence Embeddings

In [17]:
!pip install -q sentence-transformers

# We're using a pretrained transformer because then training time will be much lesser then evaluate all 3 on cosine similarity

In [18]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

print("Model loaded")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded


In [19]:
# Get the titles from each split

val_titles_1 = val_pairs["title1"].tolist()
val_titles_2 = val_pairs["title2"].tolist()

test_titles_1 = test_pairs["title1"].tolist()
test_titles_2 = test_pairs["title2"].tolist()

In [20]:
val_emb1 = model.encode(
    val_titles_1,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

val_emb2 = model.encode(
    val_titles_2,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

test_emb1 = model.encode(
    test_titles_1,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

test_emb2 = model.encode(
    test_titles_2,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", val_emb1.shape)

Batches:   0%|          | 0/52 [00:00<?, ?it/s]

Batches:   0%|          | 0/52 [00:00<?, ?it/s]

Batches:   0%|          | 0/52 [00:00<?, ?it/s]

Batches:   0%|          | 0/52 [00:00<?, ?it/s]

Embedding shape: (3304, 384)


# Calculating Cosine Similarity 

Because we put normalize_embeddings = True, we can simply calculate the dot product

In [21]:
val_embedding_similarity = np.sum(
    val_emb1 * val_emb2,
    axis=1
)

print(val_embedding_similarity[:10])

[0.31028208 0.9637793  0.34197387 0.23285732 0.8487252  0.56074536
 0.38023922 0.7697092  0.70074606 0.6088378 ]


# Find the best threshold

In [22]:
embedding_results = []

for threshold in np.arange(0.10, 0.91, 0.05):

    predictions = (
        val_embedding_similarity >= threshold
    ).astype(int)

    precision = precision_score(
        val_pairs["label"],
        predictions,
        zero_division=0
    )

    recall = recall_score(
        val_pairs["label"],
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        val_pairs["label"],
        predictions,
        zero_division=0
    )

    embedding_results.append([
        threshold,
        precision,
        recall,
        f1
    ])

embedding_results = pd.DataFrame(
    embedding_results,
    columns=["threshold", "precision", "recall", "f1"]
)

best_embedding_threshold = embedding_results.loc[
    embedding_results["f1"].idxmax(),
    "threshold"
]

print(embedding_results)
print("\nBest threshold:", best_embedding_threshold)

    threshold  precision    recall        f1
0        0.10   0.548023  0.998184  0.707573
1        0.15   0.594565  0.993341  0.743880
2        0.20   0.651497  0.987893  0.785182
3        0.25   0.724417  0.976998  0.831959
4        0.30   0.793260  0.954600  0.866484
5        0.35   0.864302  0.929177  0.895566
6        0.40   0.909091  0.889831  0.899358
7        0.45   0.952446  0.848668  0.897567
8        0.50   0.979321  0.802663  0.882236
9        0.55   0.986519  0.753027  0.854102
10       0.60   0.989610  0.691889  0.814393
11       0.65   0.998045  0.618039  0.763364
12       0.70   0.998867  0.533898  0.695858
13       0.75   0.998638  0.443705  0.614417
14       0.80   0.998294  0.354116  0.522788
15       0.85   0.997680  0.260291  0.412866
16       0.90   0.996528  0.173729  0.295876

Best threshold: 0.40000000000000013


# Final Test Score

In [23]:
test_embedding_similarity = np.sum(
    test_emb1 * test_emb2,
    axis=1
)

embedding_predictions = (
    test_embedding_similarity >= best_embedding_threshold
).astype(int)

print("MULTILINGUAL EMBEDDINGS")
print("Threshold:", best_embedding_threshold)

print("Precision:", precision_score(
    test_pairs["label"],
    embedding_predictions,
    zero_division=0
))

print("Recall:", recall_score(
    test_pairs["label"],
    embedding_predictions,
    zero_division=0
))

print("F1:", f1_score(
    test_pairs["label"],
    embedding_predictions,
    zero_division=0
))

MULTILINGUAL EMBEDDINGS
Threshold: 0.40000000000000013
Precision: 0.9228375855631612
Recall: 0.897156684815487
F1: 0.9098159509202454


# Error Analysis

In [24]:
error_analysis = test_pairs.copy()

error_analysis["similarity"] = test_char_similarity
error_analysis["prediction"] = char_predictions

false_positives = error_analysis[
    (error_analysis["label"] == 0) &
    (error_analysis["prediction"] == 1)
]

false_negatives = error_analysis[
    (error_analysis["label"] == 1) &
    (error_analysis["prediction"] == 0)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nFALSE POSITIVES")
print(
    false_positives[
        ["title1", "title2", "similarity"]
    ].sort_values("similarity", ascending=False)
    .head(5)
    .to_string(index=False)
)

print("\nFALSE NEGATIVES")
print(
    false_negatives[
        ["title1", "title2", "similarity"]
    ].sort_values("similarity")
    .head(5)
    .to_string(index=False)
)

False positives: 7
False negatives: 61

FALSE POSITIVES
                                                                                       title1                                                                                         title2  similarity
[Bright] 10pcs/lot double-side adhesive tips transparent false nails tool nail glue tool [LT]                                         FASH002 Lem Kuku Palsu | Nail Art Glue Adhesive Grosir    0.258580
     Speaker Quran Lampu LED 30 Juz Murottal + Bluetooth + Chip Memory Speaker Quran Bisa COD                                                       Speaker Bluetooth Portable MEGABASS A2DP    0.252028
                       SABUN PYARY ORIGINAL IMPORT SAUDI | 100% Original Garansi uang kembali                BELI 2 GRATIS 1PAKET / CREAM HN KECIL 15GR JAMINAN ASLI TIDAK ASLI UANG KEMBALI    0.163245
                                                               Murah meriah nyam nyam 1 lusin BOTOL MINUM MY BOTTLE BUSA [GRATIS PACKING BUB